# Clase 2 · Predecir un número y medir cuánto se equivoca

**Machine Learning** · Programa de Especialización Data Scientist · Módulo 1 · Sesión 2 de 16

Docente: **Josef Renato Rodriguez**

---

## De la Clase 1 a la Clase 2

En la Clase 1 predijimos una **etiqueta**: ¿cae en mora, sí o no? Hoy predecimos un **número**:

> ¿Cuánto vale la vivienda típica de un barrio?

Es la pregunta detrás de toda **tasación de garantías** en un crédito hipotecario. Y hoy la
respondemos con **datos reales**: el censo de California de 1990.

| | Clase 1 · clasificación | Clase 2 · regresión |
|---|---|---|
| Predice | una etiqueta (0/1) | un **número** (dólares) |
| Se mide con | exactitud | **MAE, RMSE, R², MAPE** |
| La regla trivial | decir siempre la clase más común | predecir siempre **el promedio** |
| El modelo de hoy | árbol de decisión | **regresión lineal** |

## Qué vas a hacer hoy

| Bloque | Contenido | Min |
|---|---|---|
| 1 | Las métricas de regresión, a mano y con código | 15 |
| 2 | Regresión lineal simple: de dónde sale la recta | 12 |
| 3 | Regresión múltiple: datos faltantes y variables categóricas | 13 |
| 4 | Supuestos, residuos y multicolinealidad | 12 |

Igual que en la Clase 1: completa las celdas **`TU CÓDIGO`** (donde dice `None`) y ejecuta la
**`VERIFICACIÓN`** de abajo. Las **`DEMOSTRACIÓN`** ya vienen completas.

---
## Celda 0 · Preparación y datos

El dataset **California Housing** tiene una fila por cada *block group* del censo de EE. UU. de 1990:
la unidad geográfica más pequeña con datos publicados, de 600 a 3 000 habitantes. Aquí los llamamos
**distritos**. El diccionario de datos completo está en `diccionario_california_housing.md`.

La celda intenta descargarlo de internet. Si no puede, sube `california_housing.csv` con el botón de
archivos de Colab (el ícono de carpeta, a la izquierda) y vuelve a ejecutarla.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             mean_absolute_percentage_error)
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN, GRAY = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"

def cargar():
    try:
        d = pd.read_csv(URL)
        print("Datos descargados de internet.")
    except Exception as e:
        print(f"No se pudo descargar ({type(e).__name__}). Leyendo el archivo local...")
        d = pd.read_csv("california_housing.csv")
    return d

casas = cargar()
print(f"\n{len(casas):,} distritos, {len(casas.columns)} columnas\n")
print(casas.head(5).to_string(index=False))

### Las columnas

| Columna | Qué es | Unidad |
|---|---|---|
| `longitude`, `latitude` | ubicación del distrito | grados |
| `housing_median_age` | antigüedad mediana de las viviendas | años (topado en 52) |
| `total_rooms` | total de habitaciones en el distrito | habitaciones |
| `total_bedrooms` | total de dormitorios | dormitorios (**tiene 207 vacíos**) |
| `population` | habitantes | personas |
| `households` | hogares | hogares |
| `median_income` | ingreso mediano de los hogares | **decenas de miles de USD**: 3.5 = 35 000 USD al año |
| `ocean_proximity` | cercanía al océano | categoría (5 valores) |
| `median_house_value` | **valor mediano de la vivienda. Es lo que queremos predecir** | USD (**topado en 500 001**) |

> Ojo con dos cosas que son típicas de los datos reales: **faltan valores** en `total_bedrooms`,
> y **hay topes**: el valor de la vivienda nunca pasa de 500 001, aunque en la realidad sí lo hiciera.
> Las dos vuelven más adelante.

---
# Bloque 1 · Las métricas de regresión  ·  15 min

En clasificación contábamos aciertos. Con números eso no sirve: si una vivienda vale 200 000 y
predijimos 199 000, no «fallamos». Lo que importa es **cuánto** nos equivocamos.

Todo empieza con el **residuo**: la diferencia entre lo real y lo predicho.

$$e_i = y_i - \hat{y}_i$$

| Símbolo | Se lee | Qué es |
|---|---|---|
| $y_i$ | «ye sub i» | el valor real del distrito número $i$ |
| $\hat{y}_i$ | «ye sombrero sub i» | lo que el modelo predijo para ese distrito |
| $e_i$ | «e sub i» | el residuo. Positivo = el modelo se quedó corto |
| $\bar{y}$ | «ye barra» | el promedio de los $y$ |
| $n$ | «ene» | cuántos distritos hay |

### Cinco distritos para calcular a mano

Para que las cuentas salgan en la pizarra, simplificamos: $x$ es el ingreso mediano (en decenas de
miles de USD) e $y$ el valor de la vivienda **en miles de USD**. Un modelo (en el Bloque 2 veremos
de dónde sale) predice $\hat{y} = 70 + 25 \cdot x$.

| Distrito | Ingreso $x$ | Valor real $y$ | Predicho $\hat{y}$ | Residuo $e$ |
|---|---|---|---|---|
| 1 | 2 | 120 | 120 | 0 |
| 2 | 3 | 150 | 145 | +5 |
| 3 | 4 | 160 | 170 | −10 |
| 4 | 5 | 200 | 195 | +5 |
| 5 | 6 | 220 | 220 | 0 |

### Las fórmulas

| Métrica | Fórmula | Se lee como |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum \lvert e_i \rvert$ | «en promedio me equivoco por tantos miles» |
| **MSE** | $\frac{1}{n}\sum e_i^2$ | lo mismo, pero al cuadrado: castiga mucho los errores grandes |
| **RMSE** | $\sqrt{MSE}$ | el MSE devuelto a la unidad original |
| **R²** | $1 - \frac{\sum e_i^2}{\sum (y_i - \bar{y})^2}$ | qué fracción de la variación explico, comparado con predecir el promedio |
| **MAPE** | $\frac{1}{n}\sum \frac{\lvert e_i \rvert}{y_i}$ | «en promedio me equivoco en tanto por ciento» |

### Ejercicio 1.1 — Las cinco métricas a mano

Pistas: `np.abs()`, `np.mean()`, `np.sqrt()`, `np.sum()`, y `y.mean()` para $\bar{y}$.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
x = np.array([2, 3, 4, 5, 6])                        # ingreso mediano (decenas de miles)
y = np.array([120, 150, 160, 200, 220])              # valor real (miles de USD)
y_hat = 70 + 25 * x                                  # lo que predice el modelo

e = None       # residuos: y - y_hat
mae = None
mse = None
rmse = None
r2 = None      # 1 - (suma de e²) / (suma de (y - ȳ)²)
mape = None    # como proporción (0.05 = 5 %)

if e is not None:
    print("residuos:", e)

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("MAE", mae, 4.0),
     check("MSE", mse, 30.0),
     check("RMSE", rmse, 5.4772),
     check("R²", r2, 0.9766),
     check("MAPE", mape, 0.0242)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Demostración 1.2 — scikit-learn da lo mismo

En la práctica se usan las funciones de `sklearn.metrics`. Ojo con el orden: **primero lo real,
después lo predicho**.

In [ ]:
# ── DEMOSTRACIÓN 1.2 ─────────────────────────────────────────────────────
print(f"MAE  = {mean_absolute_error(y, y_hat):.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y, y_hat)):.2f}")
print(f"R²   = {r2_score(y, y_hat):.4f}")
print(f"MAPE = {mean_absolute_percentage_error(y, y_hat):.4f}")

### Demostración 1.3 — Un solo error grande

Supón que con el distrito 3 el modelo se equivoca por **100 mil** en vez de 10. ¿Cuánto se mueve
cada métrica?

In [ ]:
# ── DEMOSTRACIÓN 1.3 ─────────────────────────────────────────────────────
e_malo = (y - y_hat).astype(float)
e_malo[2] = -100                                    # un solo distrito, un error enorme

print(f"{'':>6} | {'antes':>6} | {'con el error grande':>20}")
print(f"{'MAE':>6} | {np.mean(np.abs(y - y_hat)):>6.1f} | {np.mean(np.abs(e_malo)):>20.1f}")
print(f"{'RMSE':>6} | {np.sqrt(np.mean((y - y_hat)**2)):>6.1f} | {np.sqrt(np.mean(e_malo**2)):>20.1f}")
print()
print("El MAE se multiplica por 5.5; el RMSE, por 8.2.")
print("Al elevar al cuadrado, el RMSE castiga MUCHO más los errores grandes.")

**¿Cuál usar?** Depende de cuánto te duele un error grande:

- Si equivocarte por 100 mil es **diez veces** peor que por 10 mil → **MAE**.
- Si es **mucho más** que diez veces peor (una garantía sobrevalorada que no cubre la deuda) → **RMSE**.
- Y un diagnóstico gratis: **si el RMSE es mucho mayor que el MAE, hay unos pocos errores muy grandes.**

### Ejercicio 1.4 — La trampa del MAPE

El distrito más barato del dataset vale **15 mil USD** (exactamente, 14 999). Si el modelo predijera
**115 mil**, ¿cuál sería su error porcentual absoluto $\lvert y - \hat{y} \rvert / y$?

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
y_real, y_pred = 15, 115          # en miles de USD
error_pct = None                  # |y_real - y_pred| / y_real

if error_pct is not None:
    print(f"error porcentual = {error_pct:.0%}")

In [ ]:
# ── VERIFICACIÓN 1.4 ─────────────────────────────────────────────────────
r = [check("error porcentual del distrito más barato", error_pct, 100/15)]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.4")

---
# Bloque 2 · Regresión lineal simple  ·  12 min

La recta $\hat{y} = b_0 + b_1 x$ tiene dos números:

| Símbolo | Se lee | Qué es | En el ejemplo |
|---|---|---|---|
| $b_0$ | «be cero» | el **intercepto**: la predicción cuando $x = 0$ | 70 |
| $b_1$ | «be uno» | la **pendiente**: cuánto cambia $\hat{y}$ si $x$ sube en 1 | 25 |

De todas las rectas posibles, la regresión lineal elige la que hace **más pequeña la suma de los
residuos al cuadrado**: por eso se llama **mínimos cuadrados** (OLS). Esa recta tiene fórmula cerrada:

$$b_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad\qquad b_0 = \bar{y} - b_1 \bar{x}$$

### Ejercicio 2.1 — La recta a mano

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
dx = x - x.mean()          # desviaciones de x respecto a su promedio
dy = y - y.mean()          # desviaciones de y

b1 = None                  # suma(dx * dy) / suma(dx²)
b0 = None                  # ȳ - b1 * x̄

if b1 is not None:
    print(f"ŷ = {b0:.1f} + {b1:.1f} · x")

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("pendiente b1", b1, 25),
     check("intercepto b0", b0, 70)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

### Ejercicio 2.2 — La regla trivial de la regresión

Pasamos a los 20,640 distritos reales. El corte 70/30 ya está hecho, con una novedad:
`stratify=casas["ocean_proximity"]`. Hay una categoría, ISLAND, con **solo 5 distritos**; estratificar
asegura que cada zona aparezca en entrenamiento y prueba en la misma proporción.

Primero, la vara mínima: predecir **siempre el valor promedio** del entrenamiento.

- `pred_base`: un array del tamaño de `y_test` lleno con el promedio de `y_train`. Pista: `np.full(len(y_test), valor)`.
- `rmse_base`, `mae_base` y `r2_base` comparando `y_test` con `pred_base`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
X_todo = casas.drop(columns=["median_house_value"])
y_todo = casas["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(
    X_todo, y_todo, test_size=0.3, random_state=SEED, stratify=casas["ocean_proximity"])

pred_base = None
rmse_base, mae_base, r2_base = None, None, None

if rmse_base is not None:
    print(f"RMSE de predecir siempre el promedio = {rmse_base:,.0f} USD")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("RMSE de la regla trivial", rmse_base, 115366.9785, tol=1),
     check("MAE de la regla trivial", mae_base, 90849.0755, tol=1),
     check("R² de la regla trivial", r2_base, -0.0002)]
print()
print("2.2 OK" if all(r) else "Revisa 2.2")

### Ejercicio 2.3 — La recta con el ingreso

Ajusta una `LinearRegression` usando **solo** `median_income`. scikit-learn pide `X` como tabla
(dos corchetes: `X_train[["median_income"]]`), aunque tenga una sola columna.

- `pendiente`: `modelo.coef_[0]` · `intercepto`: `modelo.intercept_`
- `rmse_simple`, `mae_simple`, `r2_simple` en **prueba**.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
simple = None              # LinearRegression() entrenada con X_train[["median_income"]] e y_train
pendiente, intercepto = None, None
pred_simple = None         # predicciones sobre X_test[["median_income"]]
rmse_simple, mae_simple, r2_simple = None, None, None

In [ ]:
# ── VERIFICACIÓN 2.3 ─────────────────────────────────────────────────────
r = [check("pendiente", pendiente, 41650.6366, tol=1),
     check("intercepto", intercepto, 46185.9807, tol=1),
     check("RMSE en prueba", rmse_simple, 82384.2518, tol=1),
     check("R² en prueba", r2_simple, 0.4900)]
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.3")

In [ ]:
# ── DEMOSTRACIÓN 2.4 · la recta sobre los datos ──────────────────────────
if pred_simple is not None:
    fig, ax = plt.subplots()
    ax.scatter(X_test["median_income"], y_test, s=4, alpha=0.25, color=BLUE, label="distritos de prueba")
    xs = np.linspace(0.5, 15, 50)
    ax.plot(xs, intercepto + pendiente * xs, color=MAG, lw=2.5, label="recta de mínimos cuadrados")
    ax.axhline(y_train.mean(), color=GRAY, ls="--", lw=1, label="siempre el promedio")
    ax.set_xlabel("ingreso mediano (decenas de miles de USD)"); ax.set_ylabel("valor mediano de la vivienda (USD)")
    ax.legend(loc="upper left"); plt.tight_layout(); plt.show()
    print("¿Ves la línea horizontal de puntos arriba, en 500 001? Es el tope del censo. Vuelve en el Bloque 4.")

---
# Bloque 3 · Regresión lineal múltiple  ·  13 min

Con varias variables la idea es idéntica:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_p x_p$$

Cada $b_j$ se lee: **cuánto cambia la predicción si $x_j$ sube en 1, manteniendo todas las demás
constantes.** Pero antes hay un problema práctico: `LinearRegression` **no acepta valores vacíos**.

### Ejercicio 3.1 — Rellenar los datos faltantes

`total_bedrooms` tiene vacíos. La solución más simple es rellenarlos con la **mediana**. La regla de
oro: **la mediana se calcula solo con el entrenamiento**, y ese mismo número se usa para rellenar la
prueba. Si la calcularas con todo, estarías dejando que la prueba influya en el modelo (*data leakage*).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
print("Vacíos antes:", X_train["total_bedrooms"].isna().sum(), "en entrenamiento,",
      X_test["total_bedrooms"].isna().sum(), "en prueba")

mediana_dorm = None       # mediana de total_bedrooms en X_train
X_train = X_train.copy()
X_test = X_test.copy()
# rellena los vacíos de X_train["total_bedrooms"] y de X_test["total_bedrooms"] con mediana_dorm
# pista: X_train["total_bedrooms"] = X_train["total_bedrooms"].fillna(mediana_dorm)

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
r = [check("mediana de total_bedrooms en entrenamiento", mediana_dorm, 433.0),
     check_bool("ya no quedan vacíos",
                X_train["total_bedrooms"].isna().sum() + X_test["total_bedrooms"].isna().sum() == 0,
                "rellena las dos: X_train y X_test")]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

### Ejercicio 3.2 — Todas las variables numéricas

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
cols_num = ["longitude", "latitude", "housing_median_age", "total_rooms",
            "total_bedrooms", "population", "households", "median_income"]
multiple = None            # LinearRegression entrenada con X_train[cols_num]
pred_mult = None
r2_mult, mae_mult = None, None

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
r = [check("coeficiente de median_income", None if multiple is None else multiple.coef_[7], 39958.9915, tol=1),
     check("R² en prueba", r2_mult, 0.6433),
     check("MAE en prueba", mae_mult, 50588.1022, tol=1)]
print()
print("3.2 OK" if all(r) else "Revisa 3.2")

> ¿Y los coeficientes de `total_rooms`, `total_bedrooms`, `households`? Más dormitorios suben el valor,
> más habitaciones lo bajan… No intentes interpretarlos todavía: el Bloque 4 explica por qué salen así.

### Ejercicio 3.3 — La variable categórica

`ocean_proximity` es texto. Se convierte en columnas de 0 y 1 (**one-hot encoding**) y se elimina
una a propósito (`drop_first=True`), que queda como **categoría de referencia**. `get_dummies` ordena
alfabéticamente, así que la referencia es `<1H OCEAN` (a menos de una hora del océano):

| ocean_proximity | INLAND | ISLAND | NEAR BAY | NEAR OCEAN |
|---|---|---|---|---|
| <1H OCEAN | 0 | 0 | 0 | 0 |
| INLAND | 1 | 0 | 0 | 0 |
| NEAR BAY | 0 | 0 | 1 | 0 |

¿Por qué eliminar una? Si no es ninguna de las otras cuatro, ya sabemos que es `<1H OCEAN`. Dejarla
haría que una columna fuera suma exacta de las demás, y la regresión no podría separar los efectos.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
Xd_train = None     # pd.get_dummies(X_train, columns=["ocean_proximity"], drop_first=True, dtype=int)
Xd_test = None      # lo mismo con X_test

con_zona = None     # LinearRegression entrenada con Xd_train
pred_zona = None
r2_zona, mae_zona = None, None

In [ ]:
# ── VERIFICACIÓN 3.3 ─────────────────────────────────────────────────────
ok_cols = Xd_train is not None and list(Xd_train.columns) == ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN']
r = [check_bool("las columnas son las correctas (sin <1H OCEAN)", ok_cols, "usa drop_first=True y dtype=int"),
     check("coeficiente de INLAND", None if con_zona is None else con_zona.coef_[8], -41464.2308, tol=1),
     check("R² en prueba", r2_zona, 0.6513)]
print()
print("3.3 OK" if all(r) else "Revisa 3.3")

**Dos cosas para mirar con calma:**

- **INLAND** vale unos 41 mil USD **menos** que un distrito igual en todo lo demás
  pero cerca del océano. Tiene sentido, y está estimado con miles de distritos.
- **ISLAND** sale con **+171 mil USD**. Pero ese número sale de **3 distritos**. Un coeficiente
  estimado con tan pocos casos no es confiable: cambiaría muchísimo con uno más o uno menos.

> Una categoría con muy pocos casos es un problema clásico en datos reales. Opciones: juntarla con
> otra parecida, dejarla fuera, o al menos no interpretar su coeficiente.

### Demostración 3.4 — ¿Qué variable pesa más?

Los coeficientes están en unidades distintas (dólares por grado, por año, por habitación). Para
compararlos hay que **estandarizar** las variables: así cada coeficiente se lee como «cuánto cambia
el valor si la variable sube **una desviación estándar**».

In [ ]:
# ── DEMOSTRACIÓN 3.4 ─────────────────────────────────────────────────────
if Xd_train is not None:
    estandar = make_pipeline(StandardScaler(), LinearRegression()).fit(Xd_train, y_train)
    coef_std = pd.Series(estandar[-1].coef_, index=Xd_train.columns).sort_values()
    coef_std.index = [c.replace("ocean_proximity_", "zona ") for c in coef_std.index]

    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.barh(coef_std.index, coef_std.values, color=[MAG if v < 0 else BLUE for v in coef_std.values])
    ax.axvline(0, color=GRAY, lw=1)
    ax.set_xlabel("USD por cada desviación estándar")
    ax.set_title("Coeficientes estandarizados")
    plt.tight_layout(); plt.show()
    print("Las predicciones son IDÉNTICAS con o sin estandarizar:",
          np.allclose(estandar.predict(Xd_test), pred_zona))

---
# Bloque 4 · Supuestos, residuos y multicolinealidad  ·  12 min

### Demostración 4.1 — El cuarteto de Anscombe (1973)

Antes de mirar nuestros residuos, un recordatorio de por qué hay que mirarlos. Cuatro conjuntos de
11 puntos con **la misma recta y el mismo R²**.

In [ ]:
# ── DEMOSTRACIÓN 4.1 ─────────────────────────────────────────────────────
xa = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
anscombe = {
    "I":   (xa, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  (xa, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (xa, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}
print(f"{'':>4} | {'media x':>7} | {'media y':>7} | {'recta':>16} | {'R²':>5}")
fig, axes = plt.subplots(1, 4, figsize=(14, 3.3), sharey=True)
for ax, (k, (a, bb)) in zip(axes, anscombe.items()):
    a, bb = np.array(a, float), np.array(bb)
    m = LinearRegression().fit(a.reshape(-1, 1), bb)
    print(f"{k:>4} | {a.mean():>7.2f} | {bb.mean():>7.2f} | ŷ = {m.intercept_:.2f} + {m.coef_[0]:.3f}x | {m.score(a.reshape(-1,1), bb):.3f}")
    ax.scatter(a, bb, color=MAG, zorder=3)
    ax.plot([3, 20], m.predict(np.array([[3], [20]])), color=NAVY)
    ax.set_title(f"Conjunto {k}")
plt.tight_layout(); plt.show()

Solo el I es un caso para regresión lineal. El II es una curva, el III tiene un atípico, y en el IV
un único punto decide toda la pendiente. **Las métricas resumen; no muestran.**

| Supuesto | Qué dice | Cómo se ve cuando falla (en el gráfico de residuos) |
|---|---|---|
| **Linealidad** | la relación es una recta | los residuos forman una curva, no una nube |
| **Independencia** | cada error no depende del anterior | patrones en el tiempo o en el espacio |
| **Varianza constante** | el error es parecido para todos | forma de embudo |
| **Normalidad** | los errores siguen una campana | colas pesadas o puntos muy alejados |

### Demostración 4.2 — Los residuos de nuestro modelo

In [ ]:
# ── DEMOSTRACIÓN 4.2 ─────────────────────────────────────────────────────
if pred_zona is not None:
    residuos = y_test.values - pred_zona
    tope = y_test.values >= 500001
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(pred_zona[~tope], residuos[~tope], s=4, alpha=0.3, color=BLUE, label="resto")
    axes[0].scatter(pred_zona[tope], residuos[tope], s=6, alpha=0.7, color=MAG, label="valor topado en 500 001")
    axes[0].axhline(0, color=GRAY, lw=1)
    axes[0].set_xlabel("valor predicho (USD)"); axes[0].set_ylabel("residuo (real - predicho)")
    axes[0].set_title("Residuos vs predicción"); axes[0].legend(loc="upper left")
    axes[1].hist(residuos, bins=60, color=BLUE, alpha=0.8)
    axes[1].set_title("Distribución de los residuos"); axes[1].set_xlabel("residuo (USD)")
    plt.tight_layout(); plt.show()
    print(f"Distritos topados en prueba: {tope.sum()} de {len(tope):,}")
    print(f"Predicciones NEGATIVAS: {(pred_zona < 0).sum()}  ← una vivienda no puede valer menos de cero")
    print(f"Predicciones por encima del tope: {(pred_zona > 500001).sum()}")

Tres cosas que el R² de 0.651 no te contaba:

1. **Una diagonal perfecta de puntos rojos.** Son los distritos topados: su valor real está fijo en
   500 001, así que el residuo es exactamente `500 001 − predicción`. El verdadero valor era mayor;
   el censo simplemente no lo registró.
2. **La nube azul tiene forma de embudo**: el error crece con la predicción. Se viola la varianza constante.
3. **Hay predicciones negativas.** La recta no sabe que un precio no puede ser negativo.

Ninguna de las tres se ve en una métrica. Todas se ven en un gráfico.

### Ejercicio 4.3 — ¿Cuánto pesan los distritos topados?

Calcula MAE y RMSE del modelo con zona **quitando** los distritos de prueba con valor ≥ 500 001.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
sin_tope = (y_test < 500001).values          # True para los que NO están topados

mae_sin = None     # MAE usando solo y_test[sin_tope] y pred_zona[sin_tope]
rmse_sin = None

if rmse_sin is not None:
    print(f"MAE {mae_sin:,.0f} · RMSE {rmse_sin:,.0f}")

In [ ]:
# ── VERIFICACIÓN 4.3 ─────────────────────────────────────────────────────
r = [check("MAE sin topados", mae_sin, 45518.2140, tol=1),
     check("RMSE sin topados", rmse_sin, 60598.8223, tol=1)]
print()
print("4.3 OK" if all(r) else "Revisa 4.3")

### Ejercicio 4.4 — Multicolinealidad: variables que dicen lo mismo

`total_rooms`, `total_bedrooms`, `households` y `population` miden, casi, **el tamaño del distrito**.
Un distrito con el doble de hogares tiene casi el doble de dormitorios (correlación
**0.976**). Cuando varias variables explicativas se mueven juntas, la regresión no sabe a
cuál atribuir el efecto.

Se mide con el **VIF** (*variance inflation factor*): se hace una regresión de una variable contra
**todas las demás** y se calcula

$$VIF = \frac{1}{1 - R^2}$$

Si las demás no la explican nada, $R^2 = 0$ y $VIF = 1$. Regla práctica: **más de 5, mirarlo; más
de 10, es un problema.**

Calcula el VIF de `total_bedrooms` usando como explicativas las otras 7 columnas numéricas de `X_train`.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
otras = [c for c in cols_num if c != "total_bedrooms"]
r2_dorm = None     # R² de una LinearRegression que predice X_train["total_bedrooms"] con X_train[otras]
vif_dorm = None    # 1 / (1 - r2_dorm)

In [ ]:
# ── VERIFICACIÓN 4.4 ─────────────────────────────────────────────────────
r = [check("VIF de total_bedrooms", vif_dorm, 27.9117, tol=0.05),
     check_bool("el VIF supera 10", vif_dorm is not None and vif_dorm > 10)]
print()
print("4.4 OK" if all(r) else "Revisa 4.4")

### Demostración 4.5 — Qué hace la multicolinealidad

Entrenamos el mismo modelo con 5 muestras distintas de 300 distritos y predecimos siempre el mismo
**distrito típico** (la mediana de cada variable, a menos de una hora del océano).

In [ ]:
# ── DEMOSTRACIÓN 4.5 ─────────────────────────────────────────────────────
if Xd_train is not None:
    tipico = Xd_train.median().to_frame().T
    tipico[[c for c in Xd_train.columns if c.startswith("ocean")]] = 0

    print(f"{'muestra':>7} | {'b dormitorios':>13} | {'b hogares':>9} | {'b habitaciones':>14} | {'predicción':>10}")
    print("-" * 66)
    for k in range(5):
        idx = np.random.default_rng(100 + k).choice(len(Xd_train), 300, replace=False)
        m = LinearRegression().fit(Xd_train.iloc[idx], y_train.iloc[idx])
        cf = dict(zip(Xd_train.columns, m.coef_))
        print(f"{k+1:>7} | {cf['total_bedrooms']:>13.1f} | {cf['households']:>9.1f} | {cf['total_rooms']:>14.1f} | {m.predict(tipico)[0]:>10,.0f}")

- El coeficiente de **hogares cambia de signo** (de -89 a +92) y el de dormitorios se
  multiplica por cuatro. **Ninguno se puede interpretar.**
- La **predicción** del distrito típico, en cambio, se mueve poco (entre 200 y
  217 mil USD).

> La multicolinealidad **no arruina las predicciones: arruina la interpretación.**

### Demostración 4.6 — Un arreglo: de totales a razones

El problema es que las cuatro variables miden el tamaño del distrito. Lo que de verdad dice algo de una
vivienda no es cuántos dormitorios hay en total, sino **cuántos por hogar**. Reemplazamos los cuatro
totales por tres razones:

In [ ]:
# ── DEMOSTRACIÓN 4.6 ─────────────────────────────────────────────────────
def con_razones(D):
    D = D.copy()
    D["cuartos_por_hogar"] = D["total_rooms"] / D["households"]
    D["dormitorios_por_cuarto"] = D["total_bedrooms"] / D["total_rooms"]
    D["personas_por_hogar"] = D["population"] / D["households"]
    return D.drop(columns=["total_rooms", "total_bedrooms", "population", "households"])

if Xd_train is not None:
    R_train, R_test = con_razones(Xd_train), con_razones(Xd_test)
    m_raz = LinearRegression().fit(R_train, y_train)
    p_raz = m_raz.predict(R_test)
    print(f"{'':>22} | {'R²':>6} | {'MAE':>8}")
    print(f"{'con los 4 totales':>22} | {r2_zona:>6.3f} | {mae_zona:>8,.0f}")
    print(f"{'con 3 razones':>22} | {r2_score(y_test, p_raz):>6.3f} | {mean_absolute_error(y_test, p_raz):>8,.0f}")
    print("\nVIF con razones:")
    rn = ["longitude", "latitude", "housing_median_age", "median_income",
          "cuartos_por_hogar", "dormitorios_por_cuarto", "personas_por_hogar"]
    for c in rn:
        resto = [k for k in rn if k != c]
        r2c = LinearRegression().fit(R_train[resto], R_train[c]).score(R_train[resto], R_train[c])
        print(f"  {c:<24} {1/(1-r2c):>5.1f}")

El VIF de las variables de tamaño cae de ~30 a menos de 2. A cambio, el R² baja un poco
(0.651 → 0.627). Es un **trade-off** real: un modelo algo menos preciso, pero cuyos coeficientes
sí se pueden explicar a un comité de riesgos o a un regulador. La otra salida es la regularización
(Ridge), que vemos **la próxima clase**.

> ¿Y la latitud y longitud, con VIF cercano a 9? Tienen sentido juntas: una sin la otra no ubica
> nada en el mapa. No todo VIF alto se «arregla»; a veces solo se entiende.

---
# Cierre

### Los números del día (en prueba)

| | RMSE | MAE | R² |
|---|---|---|---|
| Siempre el promedio | 115 367 | 90 849 | -0.000 |
| Recta con ingreso | 82 384 | 61 556 | 0.490 |
| + 7 numéricas | 68 893 | 50 588 | 0.643 |
| + zona (one-hot) | **68 115** | **49 664** | **0.651** |
| con razones en vez de totales | 70 425 | 51 400 | 0.627 |

### Checklist de salida

- [ ] Sé calcular MAE, RMSE, R² y MAPE a mano, y sé qué dice cada una.
- [ ] Sé que RMSE ≫ MAE indica pocos errores muy grandes.
- [ ] Sé que R² compara contra predecir el promedio, y que puede ser negativo.
- [ ] Relleno faltantes con estadísticos **del entrenamiento**.
- [ ] Sé convertir una categórica con one-hot y desconfío de categorías con pocos casos.
- [ ] Siempre grafico los residuos: topes, embudos y predicciones imposibles no salen en el R².
- [ ] Detecto multicolinealidad con el VIF, y sé que afecta la interpretación, no la predicción.

### Próxima clase · Viernes 2/10

**Clase 3 · Lasso, Ridge y regresión con vectores de soporte.** Hoy los coeficientes de las
variables de tamaño se volvieron locos. La regularización les pone un freno sin tener que rediseñar
las variables a mano.